In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
import os 
os.chdir('/home/nazanin/github/ReLaFIS/')


In [3]:
import torch.nn.functional as F
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
import torch.nn as nn
from model.ReLaFIS import ReLaFIS, SklearnReLaFISWrapper
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR


In [ ]:
# config

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_size = 105
batch_size = 32
random_state = 199

lr = 0.005
max_lr = 0.01
epochs = 100

alpha = 1.0
min_alpha = 0.0
noise_evaluate = True

learning_params = dict(train_size=train_size,
                       batch_size=batch_size,
                       random_state=random_state,
                       lr=lr,
                       max_lr=max_lr,
                       epochs=epochs,
                       alpha=alpha,
                       min_alpha=min_alpha)


In [5]:
# datasetdec

from tests.tabular_small import (
    Cryotheraphy, Haberman, Heart, Glass, SegmentaitionUCI, Wine, Thyroid,
    Immunotherapy, Iris, BreastCancer, PimaDiabetes, CarEvaluation, Autism,
    Digits, DNA, MFeat, Parkinson, SyntheticGaussian, PimaDiabetesCase2,
    DigitsUCI as Digits_UCI, BCW,
)
from tests.tabular_large import AdultIncome, BankMarketing, Smoke
from tests.bio_medical import Diabetes, Colon, Leukemia, SRBCT, Madelon, ORL, Isolet
from tests.image import MNIST, FashionMNIST
from tests.uci_thresholded import (
    EnergyEfficiencyHeating, EnergyEfficiencyCooling, Airfoil, Concrete,
    ConcreteMulticlass, CCPP,
)
from tests.inequality import (
    LinearThreshold, ExponentialGrowth, LogisticGrowth, DampedOscillator, PDE,
)
from torch.utils.data import DataLoader


experiment = Haberman(session_id=random_state)
train_dataset = experiment.train_dataset(device)
test_dataset = experiment.test_dataset(device)


train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)


learning_params['train_size'] = experiment.train_numpy()[0].shape[0]
#regression = experiment.target_type == 'regression'
regression = False


In [ ]:
sns.histplot(experiment.train_numpy()[1])
sns.histplot(experiment.test_numpy()[1])

binary = len(np.unique(experiment.train_numpy()[1])) == 2
binary


In [7]:
# prepare the model - modeldec

from train.early_stop import EarlyStopping
 
model_params = {
    'in_features': experiment.train_numpy()[0].shape[1],
    'out_features': len(np.unique(experiment.train_numpy()[1])),
    'rules': 2,
    'drop_out_p': 0.3,
    'binary': binary,
}

zeta = 1.0
rank = 2
eta = 1
Xi = 1
 
entropy_coef = 0.0005

model = ReLaFIS(**model_params, dtype=torch.float32)
sk_model = SklearnReLaFISWrapper(model, device=device)
 

optimizer = optim.Adam(model.parameters(), lr=lr)
 
steps_per_epoch = len(train_loader)
scheduler = OneCycleLR(optimizer, max_lr=max_lr,
                       steps_per_epoch=steps_per_epoch, epochs=epochs)
 
early_stopping = EarlyStopping(patience=10, delta=-0.00001)
 
safe_min_alpha = max(min_alpha, 1e-6)
decay_steps = steps_per_epoch * epochs / 2
alpha_decaying = np.power(safe_min_alpha / alpha, 1.0 / decay_steps)
 
cos = nn.L1Loss()
 
if binary:
    cross = nn.BCEWithLogitsLoss()
 
    def criterion(batch_X, batch_y, outputs, reconstructed, entropy_penalty=None, alpha=1.0, entropy_coef=0.0):
        main_loss = cross(outputs.squeeze(), batch_y.squeeze())
        reconstruction_loss = cos(reconstructed, batch_X) * alpha
        if entropy_penalty is not None and entropy_coef > 0:
            entropy_loss = entropy_penalty.mean() * entropy_coef
            return main_loss + reconstruction_loss + entropy_loss
        return main_loss + reconstruction_loss
 
else:
    cross = nn.CrossEntropyLoss()
 
    def criterion(batch_X, batch_y, outputs, reconstructed, entropy_penalty=None, alpha=1.0, entropy_coef=0.0):
        main_loss = cross(outputs, batch_y.long())
        reconstruction_loss = cos(reconstructed, batch_X) * alpha
        if entropy_penalty is not None and entropy_coef > 0:
            entropy_loss = entropy_penalty.mean() * entropy_coef
            return main_loss + reconstruction_loss + entropy_loss
        return main_loss + reconstruction_loss


In [ ]:
# train


import random
import matplotlib.pyplot as plt
 
def set_random_state(seed=None):
    if seed is None:
        seed = np.random.randint(0, 2**32 - 1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return seed
 
set_random_state(random_state)
 
history = []
 
train_entropy_values = []
val_entropy_values = []
epochs_list = []
 
has_entropy_reg = hasattr(model, 'entropy_coef')
if has_entropy_reg:
    print(f"Using entropy regularization with coefficient: {model.entropy_coef}")
 
# FIX Bug 5: log interpretable params for ALL model types that support it,
# Types that expose get_interpretable_params diagnostics.
LOGGABLE_TYPES = {'relafis'}
# del
batch_idx_check = 0
 
for epoch in range(epochs):
 
    model.train()
    train_loss = 0
    train_reconstruction_loss = 0
    train_entropy_sum = 0 if has_entropy_reg else None
 
    
    stats = model.get_interpretable_params()
    stats["epoch"] = epoch
    history.append(stats)
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: {stats}")
 
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
 
        outputs = model(batch_X)
 
        if len(outputs) == 3:
            outputs, reconstructed, entropy_penalty = outputs
        else:
            outputs, reconstructed = outputs
            entropy_penalty = None
 
        if has_entropy_reg and entropy_penalty is not None:
            loss = criterion(
                batch_X, batch_y, outputs, reconstructed,
                entropy_penalty, alpha, model.entropy_coef
            )
            train_entropy_sum += entropy_penalty.mean().item() * batch_X.size(0)
        else:
            try:
                loss = criterion(batch_X, batch_y, outputs, reconstructed, None, alpha, 0.0)
            except TypeError:
                loss = criterion(batch_X, batch_y, outputs, reconstructed, alpha)
 
        loss.backward()

        # del
        if epoch == 0 and batch_idx_check==0:
            batch_idx_check += 1  # just first batch is enough
            for name, p in model.named_parameters():
                if p.grad is not None:
                    print(name, p.grad.norm().item())

        optimizer.step()
 
        try:
            scheduler.step()
        except ValueError as e:
            if "Tried to step" in str(e) and "total steps" in str(e):
                pass
            else:
                raise
 
        train_loss += loss.item() * batch_X.size(0)
        train_reconstruction_loss += torch.nn.functional.l1_loss(reconstructed, batch_X).item() * batch_X.size(0)
 
    alpha = max(safe_min_alpha, alpha * alpha_decaying)
    train_loss /= len(train_loader.dataset)
    train_reconstruction_loss /= len(train_loader.dataset)
 
    if has_entropy_reg and train_entropy_sum is not None:
        train_entropy_avg = train_entropy_sum / len(train_loader.dataset)
        train_entropy_values.append(train_entropy_avg)
    else:
        train_entropy_avg = 0.0
 
    model.eval()
    val_loss = 0
    val_entropy_sum = 0
 
    with torch.no_grad():
        for data, target in test_loader:
            output_tuple = model(data)
 
            if len(output_tuple) == 3:
                output, _, entropy_penalty = output_tuple
                if has_entropy_reg:
                    val_entropy_sum += entropy_penalty.mean().item() * data.size(0)
            else:
                output, _ = output_tuple
 
            if binary:
                loss = cross(output.squeeze(), target.squeeze())
            else:
                loss = cross(output, target.long())
 
            val_loss += loss.item() * data.size(0)
 
    val_loss /= len(test_loader.dataset)
 
    if has_entropy_reg:
        val_entropy_avg = val_entropy_sum / len(test_loader.dataset)
        val_entropy_values.append(val_entropy_avg)
    else:
        val_entropy_avg = 0.0
 
    epochs_list.append(epoch + 1)
 
    if has_entropy_reg:
        print(f'Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | Recon Loss: {train_reconstruction_loss:.4f} | Train Entropy: {train_entropy_avg:.6f} | Val Entropy: {val_entropy_avg:.6f} | Val Loss: {val_loss:.4f}')
    else:
        print(f'Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | Recon Loss: {train_reconstruction_loss:.4f} | Val Loss: {val_loss:.4f}')
 
    early_stopping(val_loss, model)
    if early_stopping.early_stop:
        print("Early stopping")
        break
 
early_stopping.load_best_model(model)
 
# ---- Entropy curve plots ----
if has_entropy_reg and train_entropy_values:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
    ax1 = axes[0]
    ax1.plot(epochs_list[:len(train_entropy_values)], train_entropy_values, 'b-', linewidth=2, label='Train Entropy', marker='o', markersize=4)
    if val_entropy_values:
        ax1.plot(epochs_list[:len(val_entropy_values)], val_entropy_values, 'r-', linewidth=2, label='Val Entropy', marker='s', markersize=4)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Entropy (per sample average)', fontsize=12)
    ax1.set_title('Entropy Minimization Progress', fontsize=14)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
 
    if train_entropy_values[0] > train_entropy_values[-1]:
        ax1.text(0.02, 0.95, '✅ Entropy Decreasing (Good!)', transform=ax1.transAxes,
                fontsize=11, color='green', bbox=dict(boxstyle="round,pad=0.3", facecolor='lightgreen'))
    else:
        ax1.text(0.02, 0.95, '⚠️ Entropy NOT Decreasing', transform=ax1.transAxes,
                fontsize=11, color='red', bbox=dict(boxstyle="round,pad=0.3", facecolor='lightcoral'))
 
    ax2 = axes[1]
    entropy_reduction = [train_entropy_values[0] - e for e in train_entropy_values]
    ax2.fill_between(epochs_list[:len(entropy_reduction)], 0, entropy_reduction, alpha=0.5, color='green')
    ax2.plot(epochs_list[:len(entropy_reduction)], entropy_reduction, 'g-', linewidth=2, marker='o', markersize=4)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Entropy Reduction', fontsize=12)
    ax2.set_title(f'Total Entropy Reduction: {entropy_reduction[-1]:.4f}', fontsize=14)
    ax2.grid(True, alpha=0.3)
 
    plt.tight_layout()
    plt.savefig('entropy_curves.png', dpi=300, bbox_inches='tight')
    plt.show()
 
    print("\n" + "="*60)
    print("ENTROPY SUMMARY")
    print("="*60)
    print(f"Initial Train Entropy: {train_entropy_values[0]:.6f}")
    print(f"Final Train Entropy:   {train_entropy_values[-1]:.6f}")
    print(f"Entropy Reduction:     {train_entropy_values[0] - train_entropy_values[-1]:.6f}")
    print(f"Reduction %:           {(1 - train_entropy_values[-1]/train_entropy_values[0])*100:.2f}%")
 
    if val_entropy_values:
        print(f"\nInitial Val Entropy:   {val_entropy_values[0]:.6f}")
        print(f"Final Val Entropy:     {val_entropy_values[-1]:.6f}")
        print(f"Val Reduction:         {val_entropy_values[0] - val_entropy_values[-1]:.6f}")
 
    if train_entropy_values[-1] < 0.01:
        print("\n⚠️ WARNING: Very low entropy (< 0.01)! Possible rule collapse. Reduce entropy_coef.")
    elif train_entropy_values[-1] > 1.0:
        print("\n⚠️ WARNING: High entropy (> 1.0)! Rules not specializing. Increase entropy_coef.")
    else:
        print(f"\n✅ Healthy entropy level: {train_entropy_values[-1]:.6f}")
 
    print(f"\n✅ Plot saved to 'entropy_curves.png'")
    print("="*60)
 
print("\n✅ Training complete!")


In [ ]:
# Evaluate Model: Classification Report, AUC, and Hyperparameters

from sklearn.metrics import classification_report, roc_auc_score
import pprint
import numpy as np

# Get raw labels (might be 1D or one‑hot)
y_train_raw = experiment.train_numpy()[1]
y_test_raw  = experiment.test_numpy()[1]

# Convert to 1D if necessary
if y_train_raw.ndim == 2:
    y_train = np.argmax(y_train_raw, axis=1)
    y_test  = np.argmax(y_test_raw, axis=1)
else:
    y_train = y_train_raw
    y_test  = y_test_raw

# Get predicted probabilities
y_train_proba = sk_model.predict_proba(experiment.train_numpy()[0])
y_test_proba  = sk_model.predict_proba(experiment.test_numpy()[0])

# For binary classification, roc_auc_score expects probability of the positive class
if binary:
    # If probabilities have 2 columns, take the second column (positive class)
    if y_train_proba.shape[1] == 2:
        y_train_proba_bin = y_train_proba[:, 1]
        y_test_proba_bin  = y_test_proba[:, 1]
    else:
        y_train_proba_bin = y_train_proba[:, 0]
        y_test_proba_bin  = y_test_proba[:, 0]
    train_auc = roc_auc_score(y_train, y_train_proba_bin)
    test_auc  = roc_auc_score(y_test,  y_test_proba_bin)
else:
    # For multiclass, check if we have enough classes in both train and test sets
    train_classes = len(np.unique(y_train))
    test_classes = len(np.unique(y_test))
    
    if train_classes > 1 and test_classes > 1:
        try:
            train_auc = roc_auc_score(y_train, y_train_proba, multi_class='ovr')
            test_auc  = roc_auc_score(y_test,  y_test_proba, multi_class='ovr')
        except ValueError:
            train_auc = float('nan')
            test_auc  = float('nan')
            print("Warning: Could not compute ROC AUC (insufficient classes)")
    else:
        train_auc = float('nan')
        test_auc  = float('nan')
        print(f"Warning: Insufficient classes for ROC AUC - Train: {train_classes}, Test: {test_classes}")

# Build result string
result = ""
result += f'Experiment Result {experiment.__class__.__name__}\n'
result += '------ train ------\n'
result += classification_report(y_train, sk_model.predict(experiment.train_numpy()[0]), digits=4)
result += f'AUC\t{train_auc}\n'
result += '------ test ------\n'
result += classification_report(y_test, sk_model.predict(experiment.test_numpy()[0]), digits=4)
result += f'AUC\t{test_auc}\n'

result += f'{pprint.pformat(model_params)}\n'
result += pprint.pformat(learning_params)

print(result)


In [ ]:
with open(f'results/{model_type}_{experiment.__class__.__name__}_{model_params["rules"]}.txt', mode='w') as f:
    f.write(result)


In [ ]:
# ROC Curve (Binary or One‑vs‑Rest) with Zoom-in on High Performance
'''import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, RocCurveDisplay
from sklearn.preprocessing import label_binarize

X_test, y_test = experiment.test_numpy()
y_score = sk_model.predict_proba(X_test)

classes = np.unique(y_test)
n_classes = len(classes)

plt.figure(figsize=(8, 8), dpi=200)

# Binary classification
if n_classes == 2:
    # Determine which column(s) to use
    if y_score.shape[1] == 2:
        # Standard case: two columns, second is probability of positive class
        y_score_bin = y_score[:, 1]
    elif y_score.shape[1] == 1:
        # Single column: assume it's probability of the positive class
        y_score_bin = y_score[:, 0]
    else:
        raise ValueError(f"Unexpected y_score shape: {y_score.shape}")

    fpr, tpr, _ = roc_curve(y_test, y_score_bin)
    roc_auc = auc(fpr, tpr)

    RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc,
                    estimator_name='Binary classifier').plot()

# Multiclass classification
else:
    # Binarize labels (One-vs-Rest)
    y_test_bin = label_binarize(y_test, classes=classes)

    for i, cls in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)

        plt.plot(fpr, tpr, label=f"Class {cls} (AUC = {roc_auc:.3f})")

    plt.legend(loc="lower right")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve (One-vs-Rest)")

# Zoom
plt.xlim([0.0, 0.025])
plt.ylim([0.99, 1.0])
plt.grid(True)

plt.show()'''
